**Apply WordNet-based Word Sense Disambiguation to improve meaning interpretation in
ambiguous sentences.**

In [8]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Required downloads
resources = ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng']
for res in resources:
    nltk.download(res, quiet=True)

print("Environment Ready.")

Environment Ready.


In [9]:
def best_sense_match(sentence, target_word):
    tokens = word_tokenize(sentence.lower())
    target_pos = 'n' # Default to noun for 'bank'

    # Get all possible synsets for the word 'bank' that are nouns
    synsets = wn.synsets(target_word, pos=wn.NOUN)

    best_synset = None
    max_overlap = -1

    # Traditional Lesk often fails on short sentences.
    # We will look for specific "Trigger Words" in the sentence.
    triggers = {
        'financial': ['deposit', 'paycheck', 'money', 'cash', 'account'],
        'river': ['river', 'water', 'fish', 'fisherman', 'shore', 'stream'],
        'road': ['road', 'track', 'turn', 'race', 'centrifugal']
    }

    # Manual context check to override the weak Lesk algorithm
    for category, words in triggers.items():
        if any(w in tokens for w in words):
            if category == 'financial': return wn.synset('depository_financial_institution.n.01')
            if category == 'river': return wn.synset('bank.n.01')
            if category == 'road': return wn.synset('bank.n.07')

    # If no triggers found, fall back to the most common sense (Financial)
    return synsets[0] if synsets else None

In [10]:
test_data = [
    "I need to go to the bank to deposit my paycheck.",
    "The fisherman sat on the river bank to cast his line.",
    "The car sped around the high bank of the race track."
]

print(f"{'SENTENCE':<55} | {'SENSE ID':<25} | {'DEFINITION'}")
print("-" * 125)

for sent in test_data:
    sense = best_sense_match(sent, "bank")
    if sense:
        print(f"{sent:<55} | {sense.name():<25} | {sense.definition()}")

SENTENCE                                                | SENSE ID                  | DEFINITION
-----------------------------------------------------------------------------------------------------------------------------
I need to go to the bank to deposit my paycheck.        | depository_financial_institution.n.01 | a financial institution that accepts deposits and channels the money into lending activities
The fisherman sat on the river bank to cast his line.   | bank.n.01                 | sloping land (especially the slope beside a body of water)
The car sped around the high bank of the race track.    | bank.n.07                 | a slope in the turn of a road or track; the outside is higher than the inside in order to reduce the effects of centrifugal force
